### Data processing and cleaning pipeline
reads in data from each study separately and saves:
- grab sample data to a pandas dataframe called grab, written to csv file called 'grab.csv'
- MPT field deployment data to a pandas dataframe called pe_tube, written to csv file called 'pe_tube.csv'

In [ ]:
# import pandas packages
import pandas as pd

from matplotlib import pyplot as plt
import numpy as np
import re

from sampling_rates import PFAS_DATA

In [ ]:
# read in site information
sites_df = pd.read_csv('sites.csv', index_col=0)

# read in station names used within temperature data sets and save assingment to dictionary
t_df = pd.read_csv(r'temperature_data.csv', index_col=0)
t_site_to_station = t_df['Station Name'].to_dict()

# Read in data from Christine Gardiner:
- paper: https://doi.org/10.1002/etc.5431
- data: https://osf.io/wzsm9/files/osfstorage
- temperature data within the bay (different data, approximation): http://www.narrbay.org/physical_data.htm
- for waste water treatment plants (WWTP) wild guess 19 +/- 5 degrees Celsius.

# downloaded files are placed in folder CG_RI and named:
- Data release_Gardiner_07_22.xlsx
- nbfsmn_daily_means_thru_2019.csv

In [ ]:
# read in grab sample data from Christe Gardiner (CC_RI)
# make sure date columns have same name and are dates
cg_ri_grab_I = pd.read_excel(
    r'CG_RI\Data release_Gardiner_07_22.xlsx', sheet_name='Table 2 Water PFAS NarrBay',
    parse_dates=['Sample date (MM/DD/YY)'], nrows=20
    )
cg_ri_grab_I.rename({
    'Site name': 'Site',
    'Sample date (MM/DD/YY)': 'Sample Date',
    }, axis=1, inplace=True)
cg_ri_grab_II = pd.read_excel(
    r'CG_RI\Data release_Gardiner_07_22.xlsx', sheet_name='Table 4 Water PFAS WWTP',
    parse_dates=['Sample date  (MM/DD/YY)'], nrows=40
    )
cg_ri_grab_II.rename({
    'Site name': 'Site',
    'Sample date  (MM/DD/YY)': 'Sample Date',
    }, axis=1, inplace=True)

# combine dataframes and reset index
cg_ri_grab = pd.concat([cg_ri_grab_I, cg_ri_grab_II], ignore_index=True)

# remove MDL rows
cg_ri_grab = cg_ri_grab.loc[cg_ri_grab['Site'] != 'Method Quantification Limit (MQL)', :]

# convert site name to site code in dataframe
site_code_name_dict = sites_df['Site name'].to_dict()
site_name_code_dict = {v: k for k, v in site_code_name_dict.items()}

cg_ri_grab['Site'].replace(site_name_code_dict, inplace=True)

# rename compound columns
cg_ri_grab.rename({
    'Perfluorobutanoic acid, in nanogram per liter': 'PFBA',
    'Perfluoropentanoic acid, in nanogram per liter': 'PFPeA',
    'Perfluorohexanoic acid, in nanogram per liter': 'PFHxA',
    'Perfluoroheptanoic acid, in nanogram per liter': 'PFHpA',
    'Perfluorooctanoic acid, in nanogram per liter': 'PFOA',
    'Perfluorononanoic acid, in nanogram per liter': 'PFNA',
    'Perfluorodecanoic acid, in nanogram per liter': 'PFDA',
    'Perfluoroundecanoic acid, in nanogram per liter': 'PFUdA',
    'Perfluorododecanoic acid, in nanogram per liter': 'PFDoA',
    'Perfluorotridecanoic acid, in nanogram per liter': 'PFTrDA',
    'Perfluorotetradecanoic acid, in nanogram per liter': 'PFTeDA',
    'Perfluorobutane sulfonic acid, in nanogram per liter': 'PFBS',
    'Perfluoropentane sulfonic acid, in nanogram per liter': 'PFPeS',
    'Perfluorohexane sulfonic acid, in nanogram per liter': 'L-PFHxS',
    'Perfluoroheptane sulfonic acid, in nanogram per liter': 'PFHpS',
    'Perfluorooctane sulfonic acid, in nanogram per liter': 'L-PFOS',
    'Perfluorononane sulfonic acid, in nanogram per liter': 'PFNS',
    'Perfluorodecane sulfonic acid, in nanogram per liter': 'PFDS',
    '4:2 Fluorotelomer sulfonate, in nanogram per liter': '4:2 FTS',
    '6:2 Fluorotelomer sulfonate, in nanogram per liter': 'G:2 FTS',
    '8:2 Fluorotelomer sulfonate, in nanogram per liter': '8:2 FTS',
    'Perfluorooctane sulfonamide, in nanogram per liter': 'FOSA',
    'N-Methyl perfluorooctane sulfonamidoacetic acid, in nanogram per liter': 'N-MeFOSAA',
    'N-Ethyl perfluorooctane sulfonamidoacetic acid, in nanogram per liter': 'N-EtFOSAA',
    }, axis=1, inplace=True)

# convert to long format table
cg_ri_grab = pd.melt(
    cg_ri_grab, id_vars=['Site', 'Sample Date'], value_name='Concentration in nanogram per liter',
                     )
# add source information
cg_ri_grab['Source'] = 'CG_RI'

In [ ]:
# read in passive sampler data from Christe Gardiner (CC_RI)
# make sure date columns have same name and are dates
cg_ri_pe_tube_I = pd.read_excel(
    r'CG_RI\Data release_Gardiner_07_22.xlsx', sheet_name='Table 3 PS PFAS NarrBay',
    parse_dates=['Sampler deployment date (MM/DD/YY)', 'Sampler recovery date  (MM/DD/YY)'], nrows=20
    )
cg_ri_pe_tube_I.rename({
    'Site name': 'Site',
    'Sampler deployment date (MM/DD/YY)': 'Deployment Date',
    'Sampler recovery date  (MM/DD/YY)': 'Recovery Date',
    }, axis=1, inplace=True)
cg_ri_pe_tube_II = pd.read_excel(
    r'CG_RI\Data release_Gardiner_07_22.xlsx', sheet_name='Table 5 PS PFAS WWTP',
    parse_dates=['Sampler deployment date (MM/DD/YY)', 'Sampler recovery date'], nrows=34
    )
cg_ri_pe_tube_II.rename({
    'Site name': 'Site',
    'Sampler deployment date (MM/DD/YY)': 'Deployment Date',
    'Sampler recovery date': 'Recovery Date',
    }, axis=1, inplace=True)

# combine dataframes and reset index
cg_ri_pe_tube = pd.concat([cg_ri_pe_tube_I, cg_ri_pe_tube_II], ignore_index=True)

# remove MDL rows
cg_ri_pe_tube = cg_ri_pe_tube.loc[cg_ri_pe_tube['Site'] != 'Method Quantification Limit (MQL)', :]

# convert site name to site code in dataframe
cg_ri_pe_tube['Site'].replace(site_name_code_dict, inplace=True)

# evaluate deployment time
cg_ri_pe_tube['Deployment Time'] = cg_ri_pe_tube['Recovery Date'] - cg_ri_pe_tube['Deployment Date']

# rename compound columns
cg_ri_pe_tube.rename({
    'Perfluorobutanoic acid, in nanogram per sampler': 'PFBA',
    'Perfluoropentanoic acid, in nanogram per sampler': 'PFPeA',
    'Perfluorohexanoic acid, in nanogram per sampler': 'PHHxA',
    'Perfluoroheptanoic acid, in nanogram per sampler': 'PFHpA',
    'Perfluorooctanoic acid, in nanogram per sampler': 'PFOA',
    'Perfluorononanoic acid, in nanogram per sampler': 'PFNA',
    'Perfluorodecanoic acid, in nanogram per sampler': 'PFDA',
    'Perfluoroundecanoic acid, in nanogram per sampler': 'PFUdA',
    'Perfluorododecanoic acid, in nanogram per sampler': 'PFDoA',
    'Perfluorotridecanoic acid, in nanogram per sampler': 'PFTrDA',
    'Perfluorotetradecanoic acid, in nanogram per sampler': 'PFTeDA',
    'Perfluorobutane sulfonic acid, in nanogram per sampler': 'PFBS',
    'Perfluoropentane sulfonic acid, in nanogram per sampler': 'PFPeS',
    'Perfluorohexane sulfonic acid, in nanogram per sampler': 'L-PFHxS',
    'Perfluoroheptane sulfonic acid, in nanogram per sampler': 'PFHpS',
    'Perfluorooctane sulfonic acid, in nanogram per sampler': 'L-PFOS',
    'Perfluorononane sulfonic acid, in nanogram per sampler': 'PFNS',
    'Perfluorodecane sulfonic acid, in nanogram per sampler': 'PFDS',
    '4:2 Fluorotelomer sulfonate, in nanogram per sampler': '4:2 FTS',
    '6:2 Fluorotelomer sulfonate, in nanogram per sampler': 'G:2 FTS',
    '8:2 Fluorotelomer sulfonate, in nanogram per sampler': '8:2 FTS',
    'Perfluorooctane sulfonamide, in nanogram per sampler': 'FOSA',
    'N-Methyl perfluorooctane sulfonamidoacetic acid, in nanogram per sampler': 'N-MeFOSAA',
    'N-Ethyl perfluorooctane sulfonamidoacetic acid, in nanogram per sampler': 'N-EtFOSAA',
    }, axis=1, inplace=True)

# convert to long format table
cg_ri_pe_tube = pd.melt(
    cg_ri_pe_tube, id_vars=['Site', 'Deployment Date', 'Recovery Date', 'Deployment Time'], value_name='Concentration in nanogram per sampler',
                     )
# save source information
cg_ri_pe_tube['Source'] = 'CG_RI'

In [ ]:
# read in temperature data for Christine Gardiner's paper
t_cg_ri = pd.read_csv(r'CG_RI\nbfsmn_daily_means_thru_2019.csv', parse_dates=['date'])
# select only relevant data (temperature)
t_cg_ri = t_cg_ri.loc[t_cg_ri['parameter']=='Temp_C', :]

In [ ]:
# initialize new temperature column
cg_ri_pe_tube[['Average Temperature in C', 'Std Temperature in C']] = np.nan

# loop over sites and extract data
for site in cg_ri_pe_tube['Site'].unique():
    site_deployments = cg_ri_pe_tube.loc[cg_ri_pe_tube['Site'] == site,:]
    t_site = t_cg_ri.loc[t_cg_ri['site']==t_site_to_station[site],:]
    # catch waste water treatment plants
    if site == 'BP' or site == 'FP':
        cg_ri_pe_tube.loc[cg_ri_pe_tube['Site'] == site, 'Average Temperature in C'] = 19
        cg_ri_pe_tube.loc[cg_ri_pe_tube['Site'] == site, 'Std Temperature in C'] = 5
        continue
    # loop over deployment dates and extract data
    for deployment_date in site_deployments['Deployment Date'].unique():
        site_start_deployments = site_deployments.loc[site_deployments['Deployment Date'] == deployment_date, :]
        t_site_start = t_site.loc[t_site['date']>= deployment_date, :]
        # loop over recovery dates and extract data
        for recovery_date in site_start_deployments['Recovery Date'].unique():
            site_start_recovery_deployments = site_start_deployments.loc[site_deployments['Recovery Date'] == recovery_date, :]
            t_site_start_recovery = t_site_start.loc[t_site_start['date']<= recovery_date]
            # add average temperature to applicable samples
            cg_ri_pe_tube.loc[site_start_recovery_deployments.index.to_list(), 'Average Temperature in C'] = np.nanmean(t_site_start_recovery['measure'].to_list())
            cg_ri_pe_tube.loc[site_start_recovery_deployments.index.to_list(), 'Std Temperature in C'] = np.nanstd(t_site_start_recovery['measure'].to_list())

# Read in data from Matthew Dunn's Cape Cod paper:
- paper: https://doi.org/10.1021/acsestwater.4c01164 
- relevant data is in the supporting information
- temperature data:
    - upstream: https://www.sciencebase.gov/catalog/item/5ae37225e4b0e2c2dd3207e5
    - midstream: https://www.sciencebase.gov/catalog/item/5ae37225e4b0e2c2dd3207e6
    - downstream: https://www.sciencebase.gov/catalog/item/5ae37225e4b0e2c2dd3207e7

# downloaded files are placed in folder MD_CC and named:
- ew4c01164_si_002.xlsx
- upstream.csv
- midstream.csv
- downstream.csv

In [ ]:
# read in grab sample data from Matthew Dunn's Cape Cod paper
md_cc_grab = pd.read_excel(
    r'MD_CC\ew4c01164_si_002.xlsx', sheet_name='S7. SPE Data',
    parse_dates=['Date Collected'], skiprows=1, nrows=18
    )
# rename sample date column
md_cc_grab.rename({
    'Date Collected': 'Sample Date',
    }, axis=1, inplace=True)
# delete na rows
md_cc_grab.dropna(axis=0, inplace=True)
# delete columns Water Type and Vol Extracted
md_cc_grab.drop(columns=['Water Type', 'Vol Extracted (mL)'], inplace=True)

# rename compound columns (remove ng/L)
new_columns = []
for elem in md_cc_grab.columns:
    if elem.endswith(' (ng/L)'):
        new_columns.append(elem[:-7])
    else:
        new_columns.append(elem)
md_cc_grab.columns = new_columns

# convert to long format table
md_cc_grab = pd.melt(
    md_cc_grab, id_vars=['Site', 'Sample Date'], value_name='Concentration in nanogram per liter',
                     )
md_cc_grab['Source'] = 'MD_CC'

# merge with CG_RI grab data
grab = pd.concat([cg_ri_grab, md_cc_grab], ignore_index=True)

In [ ]:
# read in pe tube sample data from Matt Dunns Cape Cod paper
md_cc_pe_tube = pd.read_excel(
    r'MD_CC\ew4c01164_si_002.xlsx', sheet_name='S11. Passive Sampler Data',
    parse_dates=['Date Deployed', 'Date Recovered'], skiprows=1, nrows=18
    )
# rename sample date column
md_cc_pe_tube.rename({
    'Date Deployed': 'Deployment Date',
    'Date Recovered': 'Recovery Date',
    }, axis=1, inplace=True)
# delete na rows
md_cc_pe_tube.dropna(axis=0, inplace=True)

# rename compound columns (remove ng/L)
new_columns = []
for elem in md_cc_pe_tube.columns:
    if elem.endswith(' (ng/sampler)'):
        new_columns.append(elem[:-13])
    else:
        new_columns.append(elem)
md_cc_pe_tube.columns = new_columns

# evaluate deployment time
md_cc_pe_tube['Deployment Time'] = md_cc_pe_tube['Recovery Date'] - md_cc_pe_tube['Deployment Date']

# convert to long format table
md_cc_pe_tube = pd.melt(
    md_cc_pe_tube, id_vars=['Site', 'Deployment Date', 'Recovery Date', 'Deployment Time'], value_name='Concentration in nanogram per sampler',
                     )
md_cc_pe_tube['Source'] = 'MD_CC'

In [ ]:
# read in temperature data for Matt Dunn's Cape Cod (MD_CC) paper
t_md_cc_upstream = pd.read_csv(r'MD_CC\upstream.csv', parse_dates=['DateT'], nrows=1403)
t_md_cc_midstream = pd.read_csv(r'MD_CC\midstream.csv', parse_dates=['DateT'])
t_md_cc_downstream = pd.read_csv(r'MD_CC\downstream.csv', parse_dates=['DateT'], nrows=1461)

# initialize new temperature column
md_cc_pe_tube[['Average Temperature in C', 'Std Temperature in C']] = np.nan

# loop over sites and extract data
for site in md_cc_pe_tube['Site'].unique():
    site_deployments = md_cc_pe_tube.loc[md_cc_pe_tube['Site'] == site,:]
    if site == '1-Groundwater':
        t_site = t_md_cc_upstream
    elif site == '2-River':
        t_site = t_md_cc_midstream
    elif site == '3-Estuary':
        t_site = t_md_cc_downstream
    # loop over deployment dates and extract data
    for deployment_date in site_deployments['Deployment Date'].unique():
        site_start_deployments = site_deployments.loc[site_deployments['Deployment Date'] == deployment_date, :]
        # as exact year is not available extract all data from previous years and evaluate average
        t_site_start = t_site.loc[(
            (t_site['DateT'].dt.month > deployment_date.month) |
            ((t_site['DateT'].dt.month == deployment_date.month) & (t_site['DateT'].dt.day >= deployment_date.day))
            ), :]
        # loop over recovery dates and extract data
        for recovery_date in site_start_deployments['Recovery Date'].unique():
            site_start_recovery_deployments = site_start_deployments.loc[site_deployments['Recovery Date'] == recovery_date, :]
            t_site_start_recovery = t_site_start.loc[(
                (t_site_start['DateT'].dt.month < recovery_date.month) |
                ((t_site_start['DateT'].dt.month == recovery_date.month) & (t_site_start['DateT'].dt.day <= recovery_date.day))
            ), :]
            # add average temperature to applicable samples
            md_cc_pe_tube.loc[site_start_recovery_deployments.index.to_list(), 'Average Temperature in C'] = np.nanmean(t_site_start_recovery['MeanTempC'].to_list())
            md_cc_pe_tube.loc[site_start_recovery_deployments.index.to_list(), 'Std Temperature in C'] = np.nanstd(t_site_start_recovery['MeanTempC'].to_list())

# merge with CG_RI pe tube data
pe_tube = pd.concat([cg_ri_pe_tube, md_cc_pe_tube], ignore_index=True)
    

# Read in data from Matthew Dunn's Westerly paper:
- paper: https://pubs.acs.org/doi/10.1021/acsestwater.3c00439
- relevant data is in the supporting information
- temperature data:
    - pawtucket river: https://waterdata.usgs.gov/nwis/uv?referred_module=sw&huc2_cd=01100007&index_pmcode_00010=3&sort_key=site_no&group_key=county_cd&sitefile_output_format=html_table&index_pmcode_DATETIME=2
    - westerly: https://weatherspark.com/m/26174/6/Average-Weather-in-June-in-Westerly-Rhode-Island-United-States#Figures-WaterTemperature
    - worden pond: https://lakemonster.com/lake/RI/Worden-Pond-water-temperature-1245

Concentration on sampler Ns has to be backcalculated from time averaged water conenctration, deployment time and sampling rates.

# downloaded files are placed in folder MD_WE and named:
- ew3c00439_si_001.xlsx
- t_pawtucket_river.csv
- t_westerly.csv
- t_worden_pond.csv

In [ ]:
# read in grab sample data from Matt Dunns Westerly paper
md_we_grab = pd.read_excel(
    r'MD_WE\ew3c00439_si_001.xlsx', sheet_name='Table S8. Active Sample Results',
    parse_dates=['Collection Date'], skiprows=2,
    )

# convert mixed entry to date
md_we_grab['Collection Date'].replace({'9/8/2021 Duplicate': '2021-09-08 00:00:00'}, inplace=True)
md_we_grab['Collection Date'] = pd.to_datetime(md_we_grab['Collection Date'])

# rename sample date column
md_we_grab.rename({
    'Collection Date': 'Sample Date',
    }, axis=1, inplace=True)

# rename compound columns (remove ng L-1)
new_columns = []
for elem in md_we_grab.columns:
    if elem.endswith(' (ng L-1)'):
        new_columns.append(elem[:-9])
    elif elem.endswith('(ng L-1)'):
        new_columns.append(elem[:-8])
    else:
        new_columns.append(elem)

md_we_grab.columns = new_columns
md_we_grab.rename(columns={
    'PFOS': 'L-PFOS',
    'PFHxS': 'L-PFHxS'
}, inplace=True)

# convert to long format table
md_we_grab = pd.melt(
    md_we_grab, id_vars=['Site', 'Sample Date'], value_name='Concentration in nanogram per liter',
                     )
md_we_grab['Source'] = 'MD_WE'

# merge with CG_RI grab data
grab = pd.concat([grab, md_we_grab], ignore_index=True)

In [ ]:
# read in pe tube data from Matt Dunns Westerly paper
md_we_pe_tube = pd.read_excel(
    r'MD_WE\ew3c00439_si_001.xlsx', sheet_name='Table S10. TWA Conc. ',
    parse_dates=['Deployment Date', 'Recovery Date'], skiprows=2,
    )

# read in sampling rates to back calculate Ns
md_we_sampling_rates = pd.read_excel(
    r'MD_WE\ew3c00439_si_001.xlsx', sheet_name='Table S7. Sampling Rates',index_col=0,
    skiprows=1)

# evaluate deployment time
md_we_pe_tube['Deployment Time'] = md_we_pe_tube['Recovery Date'] - md_we_pe_tube['Deployment Date']

# get temperature regime from deployment month
md_we_pe_tube['Temperature Regime'] = md_we_pe_tube['Deployment Date'].dt.month
md_we_pe_tube['Temperature Regime'] = md_we_pe_tube['Temperature Regime'].map(
    {1: 5, 2: 5, 3: 5, 4: 15, 5: 15, 6: 25, 7: 25, 8: 25, 9: 15, 10: 15, 11: 15, 12: 15},
    )
# rename compound columns (remove ng L-1) and back calculate Ns
new_columns = []
for elem in md_we_pe_tube.columns:
    if elem.endswith('(ng L-1)'):
        if elem.endswith(' (ng L-1)'):
            new_columns.append(elem[:-9])
        else:
            new_columns.append(elem[:-8])
        # back calculate Ns
        for i, row in md_we_pe_tube.iterrows():
            sampling_rate = md_we_sampling_rates.loc[row['Temperature Regime'], new_columns[-1]]
            if md_we_pe_tube.loc[i, elem] == '0..15':
                md_we_pe_tube.loc[i, elem] = 1e-3 * 0.15 *sampling_rate * row['Deployment Time'].days
            if md_we_pe_tube.loc[i, elem] not in ['<MDL', 'NR']:
                md_we_pe_tube.loc[i, elem] = 1e-3 * md_we_pe_tube.loc[i, elem] *sampling_rate * row['Deployment Time'].days
    else:
        new_columns.append(elem)
md_we_pe_tube.columns = new_columns

md_we_pe_tube.rename(columns={
    'PFOS': 'L-PFOS',
    'PFHxS': 'L-PFHxS'
}, errors="raise", inplace=True)

# convert to long format table
md_we_pe_tube = pd.melt(
    md_we_pe_tube, id_vars=['Site', 'Deployment Date', 'Recovery Date', 'Deployment Time'], value_name='Concentration in nanogram per sampler',
                     )
md_we_pe_tube['Source'] = 'MD_WE'

In [ ]:
# get temperature for worden pond -> saved data from website source code as text
# read in text
with open(r'MD_WE/t_worden_pond.txt', "r") as file:
    content = file.read()

# Regular expression to find all temp and time pairs
pattern = r':{\\"temp\\":\\"([\d.]+)\\",\\"time\\":\\"(\d+)\\"}'

# Extract matches
matches = re.findall(pattern, content)

# Separate into two lists and put in dataframe
temps = [float(temp) for temp, _ in matches]
times = [int(time) for _, time in matches]
t_md_we_worden_pond = pd.DataFrame({'Date': times, 'Temperature in F': temps})

# Optional: convert UNIX timestamps to readable dates
t_md_we_worden_pond['Date'] = pd.to_datetime(t_md_we_worden_pond['Date'], unit='s')
t_md_we_worden_pond.index = pd.to_datetime(t_md_we_worden_pond['Date'])
# linarly interpolate on daily resolution and reset index
t_md_we_worden_pond = t_md_we_worden_pond.resample('6H').interpolate().resample('1D').asfreq()
t_md_we_worden_pond['Date'] = t_md_we_worden_pond.index
t_md_we_worden_pond.index = [i for i in range(len(t_md_we_worden_pond))]
t_md_we_worden_pond['Temperature in C'] = (t_md_we_worden_pond['Temperature in F'] - 32) * (5/9)

# read in temperature from Pawtucket River in Westerly
t_md_we_pawtucket_river = pd.read_csv(r'MD_WE/t_pawtucket_river.txt', sep='\t')
# delete first row
t_md_we_pawtucket_river = t_md_we_pawtucket_river.loc[1:, :]
t_md_we_pawtucket_river.rename({
    'datetime': 'Date',
    '246213_00010': 'Temperature in C'
    }, axis=1, inplace=True)
t_md_we_pawtucket_river['Date'] = pd.to_datetime(t_md_we_pawtucket_river['Date'])
t_md_we_pawtucket_river['Temperature in C'] = pd.to_numeric(t_md_we_pawtucket_river['Temperature in C'], errors='coerce')

# read in average temperature in Westerly
t_md_we_westerly = pd.read_csv(r'MD_WE/t_westerly.csv')
# convert date to datetime format and set as index
t_md_we_westerly.index = pd.to_datetime(t_md_we_westerly['Date'])
# linarly interpolate on daily resolution and reset index
t_md_we_westerly = t_md_we_westerly.resample('D').interpolate()
t_md_we_westerly['Date'] = t_md_we_westerly.index
t_md_we_westerly.index = [i for i in range(len(t_md_we_westerly))]
t_md_we_westerly['Temperature in C'] = (t_md_we_westerly['Average T in F'] - 32) * (5/9)

In [ ]:
# initialize new temperature column
md_we_pe_tube[['Average Temperature in C', 'Std Temperature in C']] = np.nan

# loop over sites and extract data
for site in md_we_pe_tube['Site'].unique():
    site_deployments = md_we_pe_tube.loc[md_we_pe_tube['Site'] == site,:]
    if site == 1:
        t_site = t_md_we_worden_pond
    elif site in [6, 7]:
        t_site = t_md_we_westerly
    else:
        t_site = t_md_we_pawtucket_river

    # loop over deployment dates and extract data
    for deployment_date in site_deployments['Deployment Date'].unique():
        site_start_deployments = site_deployments.loc[site_deployments['Deployment Date'] == deployment_date, :]
        t_site_start_exact = t_site.loc[t_site['Date']>= deployment_date, :]
        t_site_start_season = t_site.loc[(
            (t_site['Date'].dt.month > deployment_date.month) |
            ((t_site['Date'].dt.month == deployment_date.month) & (t_site['Date'].dt.day >= deployment_date.day))
            ), :]
        
        # loop over recovery dates and extract data
        for recovery_date in site_start_deployments['Recovery Date'].unique():
            site_start_recovery_deployments = site_start_deployments.loc[site_deployments['Recovery Date'] == recovery_date, :]
            if not t_site_start_exact.empty:
                t_site_start_recovery_exact = t_site_start_exact.loc[t_site_start_exact['Date']<= recovery_date]
            if not t_site_start_exact.empty and not t_site_start_recovery_exact.empty:
                # add average temperature to applicable samples
                md_we_pe_tube.loc[site_start_recovery_deployments.index.to_list(), 'Average Temperature in C'] = np.nanmean(t_site_start_recovery_exact['Temperature in C'].to_list())
                md_we_pe_tube.loc[site_start_recovery_deployments.index.to_list(), 'Std Temperature in C'] = np.nanstd(t_site_start_recovery_exact['Temperature in C'].to_list())
            else:
                t_site_start_recovery_season = t_site_start_season.loc[(
                    (t_site_start_season['Date'].dt.month < recovery_date.month) |
                    ((t_site_start_season['Date'].dt.month == recovery_date.month) & (t_site_start_season['Date'].dt.day <= recovery_date.day))
                ), :]
                if recovery_date.month < deployment_date.month:
                    t_site_start_recovery_season = t_site.loc[(
                    (t_site['Date'].dt.month < recovery_date.month) | ((t_site['Date'].dt.month == recovery_date.month) & (t_site['Date'].dt.day <= recovery_date.day)) |
                    (t_site['Date'].dt.month > deployment_date.month) | ((t_site['Date'].dt.month == deployment_date.month) & (t_site['Date'].dt.day >= deployment_date.day))
                    ), :]
                # add average temperature to applicable samples
                md_we_pe_tube.loc[site_start_recovery_deployments.index.to_list(), 'Average Temperature in C'] = np.nanmean(t_site_start_recovery_season['Temperature in C'].to_list())
                md_we_pe_tube.loc[site_start_recovery_deployments.index.to_list(), 'Std Temperature in C'] = np.nanstd(t_site_start_recovery_season['Temperature in C'].to_list())

# merge with CG_RI grab data
pe_tube = pd.concat([pe_tube, md_we_pe_tube], ignore_index=True)

# Data from Jarod Snook made available by Jarod Snook, but also accessible via the following paper:
- https://pubs.acs.org/doi/full/10.1021/acs.est.4c14136

Two subdirectories for two campaigns
- JS_ME: Maine
- JS_DU: Duluth

In [ ]:
# read in Jarod's data from Maine
js_me_data = pd.read_csv(
    "JS_ME\js_me_data.csv", index_col=0, usecols=[0, 1, 2, 3, 4, 5, 6, 7, 8,],
    na_values=['<MDL', 'N/A'])

# delete all rows without detection in any sampler or any grab
js_me_data.dropna(subset=['Turtle_HM_PE (ng)', 'Turtle_S1_PE (ng)', 'Turtle_S3_PE (ng)', 'Turtle_S4_PE (ng)'], how='all', inplace=True)
js_me_data.dropna(subset=['S1 Grab (ng/L)',	'S3 Grab (ng/L)', 'S4 Grab  (ng/L)', 'HM Grab  (ng/L)'], how='all', inplace=True)

display(js_me_data)

# get PFAS namings to convention
js_me_data.index = js_me_data.index.str.replace('_TOF MS', '')
js_me_data = js_me_data.rename(index={
    'PFOS': 'L-PFOS', 'PFOS (Br)': 'Br-PFOS', 'PFHxS': 'L-PFHxS', 'PFHxS (Br)': 'Br-PFHxS'
    })

# extract only grab samples and rename columns
js_me_grab = js_me_data[['S1 Grab (ng/L)',	'S3 Grab (ng/L)', 'S4 Grab  (ng/L)', 'HM Grab  (ng/L)']]
js_me_grab.columns = ['S1', 'S3', 'S4', 'HM']

# convert to long format table
js_me_grab['variable'] = js_me_grab.index
js_me_grab.index = [i for i in range(len(js_me_grab))]
js_me_grab = js_me_grab.melt(id_vars=['variable'], var_name='Site', value_name='Concentration in nanogram per liter')

# add source
js_me_grab['Source'] = 'JS_ME'

# merge other grab data
grab = pd.concat([grab, js_me_grab], ignore_index=True)

# setup pe tube data
js_me_pe_tube = js_me_data[['Turtle_S1_PE (ng)', 'Turtle_S3_PE (ng)', 'Turtle_S4_PE (ng)', 'Turtle_HM_PE (ng)']]
js_me_pe_tube.columns = ['S1', 'S3', 'S4', 'HM']

# convert to long format table
js_me_pe_tube['variable'] = js_me_pe_tube.index
js_me_pe_tube.index = [i for i in range(len(js_me_pe_tube))]
js_me_pe_tube = js_me_pe_tube.melt(id_vars=['variable'], var_name='Site', value_name='Concentration in nanogram per sampler')

js_me_pe_tube['Source'] = 'JS_ME'
js_me_pe_tube['Deployment Time'] = pd.Timedelta(days=31)
js_me_pe_tube.loc[js_me_pe_tube['Site'] == 'S1', 'Deployment Time'] = pd.Timedelta(days=30)
js_me_pe_tube['Average Temperature in C'] = 2.5
js_me_pe_tube['Std Temperature in C'] = 5
js_me_pe_tube['Grab Sample Average'] = js_me_grab['Concentration in nanogram per liter'].to_list()
js_me_pe_tube['Number of Grab Samples > MDL'] = (~js_me_grab['Concentration in nanogram per liter'].isna()).astype(int).to_list()
js_me_pe_tube['Sampling Rate Measured [mL/day]'] = (
    1000 * js_me_pe_tube['Concentration in nanogram per sampler'] / (
        js_me_pe_tube['Deployment Time'].dt.days * js_me_pe_tube['Grab Sample Average']
        )).to_list()

# merge other pe tube data
pe_tube = pd.concat([pe_tube, js_me_pe_tube], ignore_index=True)

In [ ]:
# read in Jarod's data from Duluth
js_du_data = pd.read_csv(
    "JS_DU\js_du_data.csv", usecols=[0, 3, 4, 5, 6, 8, 9]
    )

# simplify sample names
js_du_data['Sample Name'].replace({
    'Du_DG_MD_01_W': 'MD',
    'Du_DG_MD_01_W_Dup': 'MD',
    'Du_DG_MR_02_W': 'MR',
    'Du_DG_RL_02_W': 'RL2',
    'Du_DG_RL_01_W': 'RL1',
    'Du_DG_RL_02_W_Dup': 'RL2'
}, inplace=True)

# delete blank samples
js_du_data = js_du_data.loc[~js_du_data['Sample Name'].str.contains('_BL_'),:]

# use
js_du_data.columns = [
    'Site', 'Concentration in nanogram per liter', 'Grab Sample Standard Deviation', 'Grab N',
    'variable', 'Deployment Time', 'Concentration in nanogram per sampler']
js_du_data.index = [i for i in range(len(js_du_data))]
js_du_data['Source'] = 'JS_DU'

js_du_data.replace({0: np.nan}, inplace=True)

# merge other grab data
js_du_grab = js_du_data[['Site', 'Concentration in nanogram per liter', 'variable', 'Source']]
grab = pd.concat([grab, js_du_grab], ignore_index=True)

js_du_data['Deployment Time'] = pd.to_timedelta(js_du_data['Deployment Time'], unit='days')
js_du_data['Average Temperature in C'] = 25
js_du_data['Std Temperature in C'] = 5
js_du_data.rename(columns={
    'Concentration in nanogram per liter': 'Grab Sample Average',
    'Grab N': 'Number of Grab Samples > MDL',
    },
    inplace=True)
js_du_data['Sampling Rate Measured [mL/day]'] = (1000 * js_du_data['Concentration in nanogram per sampler'] / (js_du_data['Deployment Time'].dt.days * js_du_data['Grab Sample Average'])).to_list()
js_du_data['Sampling Rate Uncertainty'] = (
    1000 * js_du_data['Grab Sample Standard Deviation'] * js_du_data['Concentration in nanogram per sampler'] / \
        (js_du_data['Grab Sample Average']**2 * js_du_data['Deployment Time'].dt.days)
        ).to_list()

# merge other grab data
js_du_pe_tube = js_du_data
pe_tube = pd.concat([pe_tube, js_du_pe_tube], ignore_index=True)

# Clean up and process data
- clean up grab dataframe and write to csv
- gather and convert information for pe_tube data frame (dates, compounds, temperature levels, etc.)
- calculate sampling rates from

$
R_S = \frac{N_S}{c_W \cdot \tau}
$

- write pe_tube dataframe to csv

In [ ]:
# clean final data frames and write to csv
for (index, row) in grab['variable'].items():
    if row.endswith(' '):
        grab.at[index, 'variable'] = row[:-1]  # remove trailing space
    elif row == 'G:2 FTS':
        grab.at[index, 'variable'] = '6:2 FTS'
    elif row == 'PFUdA':
        grab.at[index, 'variable'] = 'PFUnDA'
    elif row == 'PFDoA':
        grab.at[index, 'variable'] = 'PFDoDA'

grab.rename({
    'variable': 'Compound',
    }, axis=1, inplace=True)

# write data from grab samples to csv
grab.to_csv('grab_samples.csv', index=False)

# grab data with replaced strings
grab_na = grab.replace({'ND': np.nan, '<MDL': np.nan, '<MQL': np.nan, '< MQL': np.nan, 'NR': np.nan})

# clean final data frames and write to csv
for (index, row) in pe_tube['variable'].items():
    if row.endswith(' '):
        pe_tube.at[index, 'variable'] = row[:-1]  # remove trailing space
    elif row == 'PHHxA':
        pe_tube.at[index, 'variable'] = 'PFHxA'
    elif row == 'G:2 FTS':
        pe_tube.at[index, 'variable'] = '6:2 FTS'
    elif row == 'PFUdA':
        pe_tube.at[index, 'variable'] = 'PFUnDA'
    elif row == 'PFDoA':
        pe_tube.at[index, 'variable'] = 'PFDoDA'

In [ ]:

pe_tube[['Grab Sample Standard Deviation', 'Sampling Rate Uncertainty']] = np.nan

# Calculate sampling rates
for (index, row) in pe_tube.iterrows():
    # skip rows where sampling rate was evaluated beforehand (unknown sampling dates with known deployment times)
    if not pd.isna(row['Sampling Rate Measured [mL/day]']):
        continue

    # find corresponding grab data
    grab_data = grab_na.loc[(
        (grab_na['Site'] == row['Site']) & (grab_na['Sample Date'] >= row['Deployment Date']) & (grab_na['Sample Date'] <= row['Recovery Date']) & (grab_na['Compound'] == row['variable'])
        ), 'Concentration in nanogram per liter']
    grab_mean = np.nanmean(grab_data)
    grab_std = np.nanstd(grab_data)
    pe_tube.at[index, 'Grab Sample Average'] = grab_mean
    pe_tube.at[index, 'Grab Sample Standard Deviation'] = grab_std
    pe_tube.at[index, 'Number of Grab Samples > MDL'] = sum(~grab_data.isnull())

    if row['Concentration in nanogram per sampler'] in ['ND', '<MDL', '<MQL', 'NR', '< MQL']:
        pe_tube.at[index, 'Sampling Rate Measured [mL/day]'] = np.nan
        pe_tube.at[index, 'Sampling Rate Uncertainty'] = np.nan
    else:
        pe_tube.at[index, 'Sampling Rate Measured [mL/day]'] = 1000 * row['Concentration in nanogram per sampler'] / \
            (row['Deployment Time'].days * grab_mean) if grab_mean > 0 else np.nan
        pe_tube.at[index, 'Sampling Rate Uncertainty'] = 1000 * grab_std * row['Concentration in nanogram per sampler'] / \
            (grab_mean**2 * row['Deployment Time'].days) if (grab_mean>0 and grab_std>0) else np.nan
        
# rename compound
pe_tube.rename({
    'variable': 'Compound',
    }, axis=1, inplace=True)

# delete all rows with NA in sampling rate
pe_tube = pe_tube.loc[~pe_tube['Sampling Rate Measured [mL/day]'].isna(), :]

# add column for water type, functional group and carbon number
pe_tube['Water Type'] = pe_tube['Site'].astype(str).replace(sites_df['Water Type'].to_dict()).to_list()

# initialize pfas data class
pfas_data_class = PFAS_DATA()
functional_group_dict = pfas_data_class.get_functional_group_mapper()
carbon_chain_length_dict = pfas_data_class.get_number_of_carbons_mapper()
pe_tube['Functional Group'] = pe_tube['Compound'].replace(functional_group_dict).to_list()
pe_tube['Carbon Chain Length'] = pe_tube['Compound'].replace(carbon_chain_length_dict).to_list()

In [ ]:
pe_tube[['Sampling Rate Model [mL/day]', 'Std Sampling Rate Model [mL/day]', 'equilibrium half time [days]']] = np.nan

for (index, row) in pe_tube.iterrows():
    compound = row['Compound']
    temperature = row['Average Temperature in C']
    Rs, deltaRs = pfas_data_class.evaluate_sampling_rate(compound=compound, temperature_in_C=temperature)
    pe_tube.loc[index, 'Sampling Rate Model [mL/day]'] = Rs
    pe_tube.loc[index, 'Std Sampling Rate Model [mL/day]'] = deltaRs

# calculate equilibrium half time
for compound in pe_tube['Compound'].unique():
    # get sorption coefficient for compound in hlb
    [low_Ksw, Ksw, high_Ksw] = pfas_data_class.get_log_partition_coefficient_pfas_in_hlb(compound=compound)
    if not pd.isna(Ksw): 
        # extract compound data and iterate over rows
        pe_tube_compound = pe_tube.loc[pe_tube['Compound'] == compound, :]
        for index, row in pe_tube_compound.iterrows():
            # calculate and set equilibrium half timetime
            pe_tube.at[index, 'equilibrium half time [days]'] = \
                (-1) * np.log(0.5) * Ksw * pfas_data_class.sorbent_weight / row['Sampling Rate Model [mL/day]']

In [ ]:
# temperature levels
pe_tube['Temperature Level'] = np.nan
pe_tube.loc[pe_tube['Average Temperature in C'] < 10, 'Temperature Level'] = 'cold'
pe_tube.loc[pe_tube['Average Temperature in C'] > 20, 'Temperature Level'] = 'warm'
pe_tube.loc[(
    (pe_tube['Average Temperature in C'] >= 10) & (pe_tube['Average Temperature in C'] <= 20)
    ), 'Temperature Level'] = 'moderate'

# write data from pe tube samples to csv
pe_tube.to_csv('pe_tube.csv', index=False)